In [ ]:
import subprocess
import os
import pandas as pd
import numpy as np
from core.reader import read_ansys_csv, load_experimental_data

ANSYS_EXE_PATH = r"D:\Program Files\ANSYS Inc\ANSYS Student\v252\ansys\bin\winx64\MAPDL.exe" 
WORKING_DIR = os.getcwd()

def run_ansys_simulation(params):
    with open('chab_params.txt', 'w') as f:
        for p in params:
            f.write(f"{p}\n")

    input_file = "chab.mac"
    output_file = "ansys.out"
    
    cmd = [
        ANSYS_EXE_PATH, 
        "-b",
        #"-j", "opt_run",
        "-dir", WORKING_DIR, 
        "-i", input_file, 
        "-o", output_file
    ]

    try:
        subprocess.run(cmd, check=True, capture_output=True)
    except subprocess.CalledProcessError as e:
        print("Ошибка ANSYS:", e)
        return None

    try:
        df_res = read_ansys_csv("chab.csv")
        return df_res
    except Exception as e:
        print(f"Ошибка чтения CSV: {e}")
        return None

In [ ]:
real_experiment_data_folder = r'D:\Users\complex_deformations\P29\experimental_vint'
df_exp = load_experimental_data(real_experiment_data_folder)

def objective_function(params):
    """
    Считает ошибку между экспериментом и моделью Шабоша.
    params: [sig_y, c1, g1, c2, g2, c3, g3]
    """
    print(f"Simulating: {params}")
    
    df_ansys = run_ansys_simulation(params)
    
    if df_ansys is None or len(df_ansys) != len(df_exp):
        return 1e9
    
    # Считаем MSE по компонентам напряжений
    mse_zz = np.mean((df_ansys['S_ZZ'] - df_exp['S_ZZ'])**2)
    mse_tt = np.mean((df_ansys['S_TT'] - df_exp['S_TT'])**2)
    mse_tz = np.mean((df_ansys['S_TZ'] - df_exp['S_TZ'])**2)
    
    total_error = mse_zz + mse_tt + mse_tz
    print(f"Error: {total_error:.2f}")
    return total_error

In [11]:
import psutil

def kill_ansys_processes():
    for proc in psutil.process_iter():
        if proc.name() in ['ANSYS.exe', 'MAPDL.exe', 'ansys.exe']:
            proc.kill()

kill_ansys_processes()

In [ ]:
from scipy.optimize import differential_evolution

# Границы поиска для каждого параметра
# [sig_y, c1, g1, c2, g2, c3, g3]
bounds = [
    (200, 400),      # Sig_Y
    (1e4, 5e5),      # C1 (Жесткая кинематика)
    (100, 5000),     # gamma1 (Быстрое насыщение)
    (1e3, 5e4),      # C2
    (10, 500),       # gamma2
    (100, 1e4),      # C3
    (0, 100)         # gamma3 (Линейная часть)
]

result = differential_evolution(
    objective_function, 
    bounds, 
    strategy='best1bin', 
    maxiter=10,      # Количество поколений (увеличьте до 20-50 для точности)
    popsize=5,       # Размер популяции (увеличьте до 10-15)
    disp=True,
    workers=1
)

print("Оптимальные параметры найдены:")
print(result.x)

Simulating: [2.42927286e+02 3.03013682e+05 3.65750311e+03 3.07987874e+04
 3.62183192e+02 1.89010062e+03 5.65033060e+00]
Simulating: [3.94746191e+02 2.85590946e+05 4.00879563e+02 1.49766452e+03
 2.03903363e+02 6.81940900e+03 9.18597999e+01]
Simulating: [3.89343856e+02 2.00784435e+05 4.35404096e+03 4.30727999e+04
 4.64004719e+02 5.82044048e+03 2.87653383e+01]
Simulating: [2.78754088e+02 2.47814334e+05 7.00459331e+02 3.41251365e+04
 1.72281229e+01 3.63506358e+03 8.85340445e+01]
Simulating: [  349.89580365 18554.95199092  1955.1642337  21218.76745009
   409.88696644  7443.88175523    74.06912298]
Simulating: [3.64219309e+02 1.47458039e+05 1.70066045e+03 6.32048900e+03
 4.47992782e+02 5.48721784e+03 8.95471525e+01]
Simulating: [3.36241553e+02 1.81357235e+05 4.18536663e+03 2.88786266e+04
 1.12327175e+02 2.23350516e+03 1.68054945e+01]
Simulating: [2.19930844e+02 4.06301923e+05 1.27893764e+03 4.47351548e+04
 5.08791735e+01 7.57499697e+03 8.65033826e+00]
Simulating: [3.31021480e+02 8.45632585e+